# Analyzing Distortion in LMArena Setting

## Set-up

In [2]:
from datasets import load_dataset

ds = load_dataset("lmarena-ai/arena-human-preference-140k")

In [3]:
train_data = ds['train']

Import the utils that I wrote.

In [4]:
import utils_3 as ut

## Filtering Data via Vectors

In [5]:
ds_keys = [
    ('is_code',),
    ('category_tag', 'creative_writing_v0.1', 'creative_writing'),
    ('category_tag', 'criteria_v0.1', 'complexity'),
    ('category_tag', 'criteria_v0.1', 'creativity'),
    ('category_tag', 'criteria_v0.1', 'domain_knowledge'),
    ('category_tag', 'criteria_v0.1', 'problem_solving'),
    ('category_tag', 'criteria_v0.1', 'real_world'),
    ('category_tag', 'criteria_v0.1', 'specificity'),
    ('category_tag', 'criteria_v0.1', 'technical_accuracy'),

    ('category_tag', 'math_v0.1', 'math'),
    ('category_tag', 'if_v0.1', 'if'),
]

In [6]:
vector_len = len(ds_keys)

In [7]:
import numpy as np

def bit_strings(n):
    x = np.arange(2**n, dtype=np.uint32)[:, None]
    return ((x >> np.arange(n - 1, -1, -1)) & 1)

In [8]:
vector_keys = bit_strings(vector_len)
print(vector_keys)

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 1]
 [0 0 0 ... 0 1 0]
 ...
 [1 1 1 ... 1 0 1]
 [1 1 1 ... 1 1 0]
 [1 1 1 ... 1 1 1]]


In [9]:
all_candidates = set(train_data['model_a']).union(set(train_data['model_b']))
all_candidate_idxs = {cand : i for i, cand in enumerate(all_candidates)}

In [10]:
from tqdm import tqdm
import numpy as np

In [11]:
def process(ds_dict, mask, all_candidate_idxs):
    w = np.asarray(ds_dict["winner"])[mask]
    valid_mask = (w == "model_a") | (w == "model_b")

    w = w[valid_mask]
    a = np.asarray(ds_dict["model_a"])[mask][valid_mask]
    b = np.asarray(ds_dict["model_b"])[mask][valid_mask]

    a_idx = np.fromiter((all_candidate_idxs[x] for x in a), dtype=np.int64, count=len(a))
    b_idx = np.fromiter((all_candidate_idxs[x] for x in b), dtype=np.int64, count=len(b))

    model_a_wins = (w == "model_a")
    winners = np.where(model_a_wins, a_idx, b_idx)
    losers = np.where(model_a_wins, b_idx, a_idx)

    return winners, losers

## Computing Distortion and Misspecification Metrics

In [12]:
n_items = len(all_candidate_idxs)
key_vector = vector_keys[5]

In [13]:
def filter_fast(ds_dict, keys, key_vector, language="en", show_progress=True):
    if language is not None:
        mask = np.asarray(ds_dict["language"]) == language
    else:
        mask = np.ones(len(ds_dict), dtype=bool)

    iterator = zip(keys, key_vector)
    if show_progress:
        iterator = tqdm(iterator, total=len(keys))

    for path, target in iterator:
        values = ds_dict
        for k in path:
            values = values[k]
        values = np.asarray(values)
        mask &= (values == target)

    return mask

In [14]:
print('filtering')
mask = filter_fast(train_data, ds_keys, key_vector)

filtering


100%|██████████| 11/11 [01:04<00:00,  5.89s/it]


In [15]:
print('processing')
winners, losers = process(train_data, mask, all_candidate_idxs)

processing


In [16]:
len(losers)

6

In [17]:
print('fitting')
r_hat, result = ut.fit_bradley_terry(winners, losers, n_items)

fitting


In [ ]:
len(r_hat)

53

In [19]:
true_ranking = np.argsort(-r_hat)
borda_scores, ranking = ut.borda_from_population_utilities(r_hat[None])
distortion, k = ut.leaderboard_dist(ranking, true_ranking, r_hat)
misspec = ut.misspecification_error(winners, losers, r_hat)

In [20]:
distortion

1.0

In [21]:
misspec

2.8545740154324785e-05

## Actual Computations

In [ ]:
pop_utilities = []
absolute_subsection_sizes = []
beta = 1.0

misspec_errors = {}

for i, vector_key in tqdm(enumerate(vector_keys)):

    mask = filter_fast(train_data, ds_keys, vector_key, language=None)
    winners, losers = process(train_data, mask, all_candidate_idxs)
    r_hat, result = ut.fit_bradley_terry(winners, losers, n_items, beta=beta)

    misspec_error = ut.misspecification_error(winners, losers, r_hat, beta=beta)
    pop_utilities.append(r_hat)
    absolute_subsection_sizes.append(len(winners))

    assert len(winners) == len(losers)

    misspec_errors[i] = misspec_error

100%|██████████| 11/11 [00:55<00:00,  5.06s/it]
70it [3:08:35, 63.11s/it]

In [ ]:
pop_utilities = np.stack(pop_utilities)
print(pop_utilities.shape)

avg_utils = pop_utilities.sum(axis=0)
true_ranking = np.argsort(-avg_utils)

In [ ]:
borda_scores, ranking = ut.borda_from_population_utilities(pop_utilities, beta=beta)
distortion, k = ut.leaderboard_dist(true_ranking=true_ranking, ranking=ranking, avg_utils=avg_utils)

## Comparison with Population Utilities Scaled by Sizes